# 01 · Explore the data

Load one subject's trials and look at the behaviour. This is a starting point —
copy a cell and adapt it for your own analysis.


In [ ]:
# === SETUP — run this first, every time you open a fresh Colab runtime ===
# Clones the project and makes `from observers import api` work. The installable
# package lives in the `hierarchical/` subdirectory, so we cd there before install.
import os, sys
REPO   = "NeuroMatch_2026_Behaviour"
BRANCH = "model-verification"
URL    = "https://github.com/salmaelhanchi/NeuroMatch_2026_Behaviour.git"
if "google.colab" in sys.modules:
    if not os.path.isdir(REPO):
        !git clone -q --branch {BRANCH} {URL}
    %cd {REPO}/hierarchical
    !git pull -q
    !pip install -q -e .
    print("setup complete — latest code loaded from", os.getcwd())
else:
    # Local Jupyter: assume the notebook is launched from within the checkout.
    # Walk up to the `hierarchical/` dir that contains the `observers` package.
    here = os.path.abspath(os.getcwd())
    while here != "/" and not os.path.isdir(os.path.join(here, "observers")):
        here = os.path.dirname(here)
    os.chdir(here); sys.path.insert(0, here)
    print("local run — repo root:", here, "| observers present:", os.path.isdir("observers"))


### Load a subject
`api.load_subject(id)` returns a pandas DataFrame of that subject's trials in chronological order.


In [ ]:
from observers import api
trials = api.load_subject(1)
print(trials.shape)
trials.head()


Columns:
- `motion_direction` — the true stimulus direction (1–360°)
- `motion_coherence` — 0.06 / 0.12 / 0.24 (sensory strength; higher = clearer)
- `prior_std` — the block's prior width in degrees (10 / 20 / 40 / 80)
- `estimate_dir` — the subject's reported direction (1–360°)


### Circular estimation error
The signed error between report and stimulus, wrapped to (−180°, 180°].


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def circular_error(estimate, target):
    return (estimate - target + 180) % 360 - 180

trials["error"] = circular_error(trials.estimate_dir, trials.motion_direction)

# error spread by coherence — noisier (low-coherence) trials should be wider
fig, ax = plt.subplots(figsize=(8, 4))
for c in sorted(trials.motion_coherence.unique()):
    ax.hist(trials[trials.motion_coherence == c].error, bins=60,
            histtype="step", density=True, label=f"coherence {c}")
ax.set_xlabel("estimation error (deg)")
ax.set_ylabel("density")
ax.legend()
ax.set_title("Subject 1 — estimation error by coherence")
plt.show()


### Where to go next
- Compare **early vs late** trials within a block (behavioural learning — task 6):
  build a within-block trial index and see if error shrinks or bias toward the
  prior (225°) grows.
- Split by `prior_std` to see how prior width changes the pull toward 225°.
